# ES vs MongoDB — 리뷰 키워드 추출 벤치마크

* 담당: 김준영(Mongo)·남현식(ES) · 요구사항 **#3 "왜 Elasticsearch인가"** 의 근거 자료
* 한 줄 질문: **"Steam 리뷰에서 게임별 핵심 키워드를 뽑는 일을, MongoDB만으로 할 때와 Elasticsearch로 할 때 무엇이 다른가?"**
* 이 프로젝트에서 ES를 쓰는 이유는 *리뷰 키워드 추출(⑤)* 입니다. 그래서 비교도 **그 일 그대로** 합니다. (게임 검색은 지금 MariaDB로 하므로 ES 도입 이유로 쓰지 않습니다.)

### 이 비교가 우리 데이터 흐름의 어디인가

| 단계 | 내용 | 이 노트북에서 |
| --- | --- | --- |
| 수집·전달 | Steam 리뷰 → Kafka `raw.steam` (kind=review) | 이미 수집됨 |
| 저장 | Kafka Connect → **MongoDB `gamenuri_raw.reviews`** (원본 봉투 `payload`) | **비교 대상 A** |
| 동기화 | Monstache → **ES `reviews`** (`review.nouns` = nori 명사만) | **비교 대상 B** |
| 활용 | 게임별 키워드 TOP → (하루 1회) AI 요약 → `refined.review_summary` | 같은 일을 A·B로 각각 해 봄 |

### 비교하는 세 가지 방법
| 방법 | 하는 일 | 형태소 분석 |
| --- | --- | --- |
| **ES** | `terms` 집계 on `review.nouns` (nori, 명사만·2글자 이상·불용어 제외) | ✔ (인덱스 안에서) |
| **Mongo(집계)** | 파이프라인 `$regexFindAll` → `$unwind` → `$group`. **Mongo 안에서 끝낼 수 있는 최선**. ES와 같은 불용어를 적용 | ✘ (한글·영문 덩어리로 자름) |
| **Mongo+Kiwi** *(선택)* | 리뷰를 꺼내 와 Python 형태소 분석기(kiwipiepy)로 명사를 추출 | ✔ (DB 밖에서) |

> 세 방법 모두 **"그 키워드가 들어 있는 리뷰 수(문서 빈도)"** 로 순위를 매깁니다. ES `terms` 집계가 그렇게 세기 때문입니다.
> 이 노트북은 **읽기만 합니다.** 인덱스·데이터·설정을 만들거나 지우지 않습니다.

### 분석 순서
1. 환경 → 2. **가설 설정** → 3. 연결·데이터 점검 → 4. 추출 함수 → 5. 측정 실행 → 6. 탐색(EDA) → 7. **통계 검정** → 8. 민감도 → 9. **결론·제안**

### 준비 (한 번만)
```bash
pip install pymongo "elasticsearch>=8.13,<9" pandas numpy scipy seaborn matplotlib statsmodels jupyterlab
pip install kiwipiepy        # 선택. 없으면 'Mongo+Kiwi'는 건너뜁니다 (Python 버전에 따라 설치가 안 될 수 있음)
```
접속 정보는 **환경 변수**로 넣습니다 (노트북·코드에 쓰지 않습니다). 이름은 `shared/gamenuri_common/config.py`와 같습니다.
```powershell
$env:MONGO_URI="mongodb://<계정>:<비밀번호>@<mongo 호스트>:27017/?authSource=admin"
$env:MONGO_DB="gamenuri_raw"
$env:ES_URL="http://<elk 호스트>:9200"
$env:ES_USER="elastic"          # 비밀번호(ES_PASSWORD)는 비워 두면 실행할 때 입력창이 뜹니다
```
> ⚠ 운영 중인 DB에 질의를 보냅니다. **사용자가 적은 시간대**에 실행하고, 느려지면 2장의 `N_GAMES`·`REPEATS`를 줄이세요.

## 1. 환경 설정

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# 라이브러리: pymongo/elasticsearch(접속), pandas/numpy(처리), scipy/statsmodels(통계), matplotlib/seaborn(시각화)
import os, re, json, time, getpass, math, statistics
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from scipy import stats
from statsmodels.stats.multitest import multipletests

%matplotlib inline

In [ ]:
# 그래프의 한글이 깨지지 않도록 한글 폰트를 지정합니다. (Windows: 맑은 고딕, Mac: AppleGothic)
candidates = ['Malgun Gothic', 'AppleGothic', 'NanumGothic', 'Noto Sans CJK KR', 'Noto Sans CJK JP']
available = {f.name for f in fm.fontManager.ttflist}
font_name = next((c for c in candidates if c in available), None)

plt.rcParams['font.family'] = font_name or 'sans-serif'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 110
sns.set_style('whitegrid', {'font.family': plt.rcParams['font.family']})
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', lambda v: f'{v:,.3f}')

BLUE, ORANGE, GRAY, GREEN = '#2a78d6', '#eb6834', '#a3a29d', '#2f9e6e'   # ES=파랑, Mongo=주황, Kiwi=초록
print('사용 폰트:', font_name)

from matplotlib.ticker import FuncFormatter
def plain_log(ax, axis='y'):
    """로그축 눈금을 10⁻¹ 대신 0.1, 1, 10, 100 으로 표시 (한글 폰트에서 위첨자가 깨지는 것을 피함)."""
    def minor(v, _):                       # 2, 3, 5 배 지점만 작게 표시
        lead = round(v / 10 ** math.floor(math.log10(v))) if v > 0 else 0
        return f'{v:g}' if lead in (2, 5) else ''
    target = ax.yaxis if axis == 'y' else ax.xaxis
    target.set_major_formatter(FuncFormatter(lambda v, _: f'{v:g}'))
    target.set_minor_formatter(FuncFormatter(minor))

## 2. 가설 설정

데이터를 보기 **전에** 가설과 판정 기준을 정합니다. 결과가 가설과 달라도 그대로 보고합니다.
단위는 **게임 1개(= 키워드 추출 1회)** 이고, 같은 게임을 방법끼리 짝지어 비교합니다(짝 비교).

| # | 가설 (H1: 우리가 기대하는 것) | 귀무가설 (H0) | 통계 도구 | 실무 기준 ("의미 있다"의 조건) |
| --- | --- | --- | --- | --- |
| H1 | **품질** — ES(nori)는 Mongo(집계)보다 *조사·어미가 붙은 가짜 키워드*가 적다 | 두 방법의 잡음 비율에 차이가 없다 | Shapiro-Wilk로 정규성 확인 → **Wilcoxon 부호순위**(단측), 부트스트랩 신뢰구간, 순위 상관 효과크기 | 잡음 비율이 **20%p 이상** 낮음 |
| H2 | **속도** — 같은 게임의 키워드 추출은 ES가 Mongo(집계)보다 빠르다 | 시간 비율의 중앙값이 1이다 | **Wilcoxon 부호순위**(단측, 로그 비율), 부트스트랩 신뢰구간 | Mongo가 **1.5배 이상** 느림 |
| H3 | **규모** — 리뷰가 많은 게임일수록 Mongo(집계) 시간이 ES보다 더 가파르게 늘어난다 | 두 방법의 (로그-로그) 기울기 차이가 0이다 | **로그-로그 회귀 기울기**, **Spearman**, 게임 단위 부트스트랩으로 기울기 차이의 신뢰구간 | 기울기 차이 **0.2 이상** |
| (참고) H4 | *(Kiwi가 있을 때만, 탐색용)* Mongo+Kiwi의 품질은 ES와 비슷하다 | – | Wilcoxon 양측 + 동등 범위(±10%p) 신뢰구간 | 신뢰구간이 ±10%p 안 |

* **유의수준 α = 0.05, H1~H3 세 개는 Holm 보정**. 통계적으로 유의해도 *실무 기준*에 못 미치면 "의미 없음"으로 판정합니다.
* **잡음 비율** = 상위 K개 키워드 중 "조사·어미가 붙은 어절"로 의심되는 비율. 정의는 4장에서 코드로 보입니다(같은 게임 어휘에 '어간'이 따로 존재하면 의심). Kiwi가 있으면 **독립된 두 번째 판정**(명사만으로 분석되는지)도 같이 봅니다.
* 속도는 **이 PC에서 잰 왕복 시간**(네트워크 포함)입니다. 두 DB에 같은 조건이 걸리지만 "서버 내부 시간"은 아닙니다.

### 분석 설정값 (한 곳에서 관리)

In [ ]:
# ---- 위치 ----
RESULT_DIR = Path(os.environ.get('GAMENURI_RESULT_DIR', '../results/benchmark'))
RESULT_DIR.mkdir(parents=True, exist_ok=True)

# ---- 대상 ----
LANG = 'koreana'            # 한국어 리뷰만 (수집기가 koreana로 모음)
ES_INDEX, MONGO_COLL = 'reviews', 'reviews'
N_GAMES = 25                # 측정할 게임 수. 리뷰 수가 작은 것~큰 것이 고루 들어가게 뽑음(H3용)
MIN_REVIEWS = 300           # 이보다 리뷰가 적은 게임은 제외
N_CANDIDATES = 300          # 후보로 볼 게임 수 상한

# ---- 측정 ----
TOP_K = 20                  # 기본 키워드 개수 (민감도: 10·20·50)
TOP_K_MAX = 50              # 한 번에 받아 두고 10·20으로 잘라 비교
VOCAB_K = 300               # '어간 존재' 판정용 어휘 크기 (측정 시간에는 넣지 않음)
WARMUP, REPEATS = 1, 5      # 버리는 첫 실행 1회 + 기록 5회 → 중앙값
MONGO_TIMEOUT_MS = 60_000   # 한 질의 상한(운영 DB 보호)
MAX_FETCH = 20_000          # Mongo+Kiwi: 게임당 가져올 리뷰 상한
# ES 인덱스 매핑의 review_stop 과 같은 불용어 (Mongo 집계에도 똑같이 적용해 공정하게 비교)
STOPWORDS = ['게임', '겜', '진짜', '그냥', '정도', '생각', '사람', '하나', '부분', '느낌', '이번', '작품']

# ---- 통계 ----
ALPHA = 0.05
PRACTICAL_NOISE = 0.20      # H1: 잡음 비율 20%p 이상 차이
PRACTICAL_SPEED = 1.5       # H2: 1.5배 이상 차이
PRACTICAL_SLOPE = 0.2       # H3: 기울기 차이 0.2 이상
EQUIV = 0.10                # H4: 동등 범위
N_BOOT, SEED = 2000, 42
rng = np.random.default_rng(SEED)

## 3. 연결과 데이터 점검

벤치마크가 공정하려면 **두 저장소에 같은 데이터**가 있어야 합니다(Monstache 동기화가 끝나 있어야 함). 이를 먼저 확인합니다.

In [ ]:
# 접속 정보는 환경 변수에서만 읽습니다. 비밀번호는 화면에 출력하지 않습니다.
MONGO_URI = os.environ.get('MONGO_URI', 'mongodb://localhost:27017')
MONGO_DB = os.environ.get('MONGO_DB', 'gamenuri_raw')
ES_URL = os.environ.get('ES_URL', 'http://localhost:9200')
ES_USER = os.environ.get('ES_USER', '')
ES_PASSWORD = os.environ.get('ES_PASSWORD', '')
if ES_USER and not ES_PASSWORD:
    ES_PASSWORD = getpass.getpass('ES 비밀번호 (화면에 표시되지 않음): ')

def host_only(uri: str) -> str:
    """주소에서 계정·비밀번호를 빼고 호스트만 보여 줍니다."""
    return uri.split('://')[-1].split('@')[-1].split('/')[0].split('?')[0]

try:
    from pymongo import MongoClient
    from elasticsearch import Elasticsearch
except ImportError as e:
    raise SystemExit(f'패키지가 없습니다: {e}. 위 "준비" 셀의 pip 명령을 먼저 실행하세요.')

mongo = MongoClient(MONGO_URI, serverSelectionTimeoutMS=8000)
coll = mongo[MONGO_DB][MONGO_COLL]
es = Elasticsearch(ES_URL, basic_auth=(ES_USER, ES_PASSWORD) if ES_USER else None, request_timeout=60)

print('MongoDB :', host_only(MONGO_URI), '/', MONGO_DB, '/', MONGO_COLL)
print('ES      :', host_only(ES_URL), '/', ES_INDEX)
mongo.server_info()          # 연결 확인 (실패하면 여기서 오류)
es.info()
print('두 저장소 연결 OK')

In [ ]:
# 3-1. 점검: 문서 수, 필요한 필드/인덱스가 있는가
n_mongo_all = coll.estimated_document_count()
n_es_all = es.count(index=ES_INDEX)['count']
mapping = es.indices.get_mapping(index=ES_INDEX)
mapping = getattr(mapping, 'body', mapping)          # 응답 객체 → 일반 dict
props = list(mapping.values())[0]['mappings']['properties']
has_nouns = 'nouns' in props.get('review', {}).get('fields', {})

idx_info = coll.index_information()
mongo_has_index = any(any(k[0] == 'payload.steam_appid' for k in v['key']) for v in idx_info.values())

check = pd.DataFrame([
    ['Mongo 전체 리뷰 문서 수(추정)', f'{n_mongo_all:,}'],
    ['ES 전체 리뷰 문서 수', f'{n_es_all:,}'],
    ['ES에 review.nouns 필드(nori 명사)', '있음' if has_nouns else '없음 ⚠'],
    ['Mongo에 payload.steam_appid 인덱스', '있음' if mongo_has_index else '없음 ⚠ (전체 훑기 → Mongo가 불리하게 측정됨)'],
], columns=['점검 항목', '결과'])
display(check)
assert has_nouns, 'ES reviews에 review.nouns 필드가 없습니다. infra/monstache/README.md의 인덱스 생성 절차를 확인하세요.'
print('\nMongo 인덱스 목록:', {k: v['key'] for k, v in idx_info.items()})

**읽는 법.** Mongo에 `payload.steam_appid` 인덱스가 **없으면** 게임 하나를 집계할 때마다 컬렉션을 통째로 훑어 Mongo가 실제보다 불리하게 나옵니다. 그 경우 결론의 속도 부분에 *"인덱스 없는 상태 기준"* 이라고 꼭 적습니다. (이 노트북은 인덱스를 만들지 않습니다. 만들지는 팀이 결정합니다.)

In [ ]:
# 3-2. 측정할 게임 고르기: ES에서 한국어 리뷰가 MIN_REVIEWS 이상인 게임 후보를 받아,
#      리뷰 수 순으로 줄 세운 뒤 N_GAMES개를 고르게 뽑는다 (작은 게임~큰 게임이 모두 들어가야 H3를 볼 수 있음).
r = es.search(index=ES_INDEX, size=0,
              query={'term': {'language': LANG}},
              aggs={'g': {'terms': {'field': 'steam_appid', 'size': N_CANDIDATES, 'min_doc_count': MIN_REVIEWS}}})
cand = pd.DataFrame([(b['key'], b['doc_count']) for b in r['aggregations']['g']['buckets']],
                    columns=['steam_appid', 'n_es']).sort_values('n_es').reset_index(drop=True)
print(f'후보 게임 {len(cand)}개 (리뷰 {MIN_REVIEWS}건 이상)')
assert len(cand) >= 5, '후보가 너무 적습니다. MIN_REVIEWS를 낮추거나 데이터 동기화를 확인하세요.'

pick = np.linspace(0, len(cand) - 1, min(N_GAMES, len(cand))).round().astype(int)
games = cand.iloc[sorted(set(pick))].reset_index(drop=True)

# 두 저장소 건수 일치 확인 (Monstache 동기화가 끝났는가)
games['n_mongo'] = [coll.count_documents({'payload.steam_appid': int(a), 'payload.language': LANG},
                                         maxTimeMS=MONGO_TIMEOUT_MS) for a in games['steam_appid']]
games['diff_pct'] = (games['n_mongo'] - games['n_es']) / games['n_es'] * 100
display(games.describe().T[['min', '50%', 'max']])
bad = games[games['diff_pct'].abs() > 1]
print(f'\nMongo와 ES 건수가 1% 넘게 다른 게임: {len(bad)}개 / {len(games)}개')
if len(bad):
    display(bad)
    print('→ 동기화가 덜 끝났을 수 있습니다. 많으면 Monstache 상태를 확인한 뒤 다시 실행하세요.')

In [ ]:
# 3-3. 시각화: 측정 대상 게임의 리뷰 수 분포
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.bar(range(len(games)), games['n_es'], color=BLUE)
ax.set_yscale('log'); plain_log(ax)
ax.set_xlabel('측정 대상 게임 (리뷰 수 순)')
ax.set_ylabel('한국어 리뷰 수 (로그)')
ax.set_title(f'측정 대상 {len(games)}개 게임의 리뷰 수')
plt.tight_layout(); plt.savefig(RESULT_DIR / 'bench_games.png', dpi=130); plt.show()

#### 차트 해설 / 이 차트로 알 수 있는 것
* 세로축이 **로그**이므로 막대 높이가 한 칸 오르면 10배입니다.
* 작은 게임부터 큰 게임까지 고르게 들어 있어야 7장의 **H3(규모에 따른 시간 증가)** 를 볼 수 있습니다. 한쪽에 몰려 있으면 H3는 "표본 범위 부족"으로 봐야 합니다.

## 4. 키워드 추출 함수 (세 방법)

세 방법 모두 *"게임 하나의 한국어 리뷰에서, 그 단어가 들어 있는 리뷰 수 기준 상위 K개"* 를 돌려줍니다.

In [ ]:
# ---- 방법 1: ES. nori로 이미 쪼개 둔 review.nouns 필드를 terms 집계 (캐시를 꺼서 매번 실제로 계산) ----
def es_keywords(appid: int, k: int):
    r = es.search(index=ES_INDEX, size=0, request_cache=False,
                  query={'bool': {'filter': [{'term': {'steam_appid': appid}}, {'term': {'language': LANG}}]}},
                  aggs={'kw': {'terms': {'field': 'review.nouns', 'size': k}}})
    return [(b['key'], b['doc_count']) for b in r['aggregations']['kw']['buckets']]

# ---- 방법 2: Mongo 집계. 형태소 분석이 없으니 한글·영문·숫자 덩어리(2글자 이상)로 자른다.
#      리뷰마다 중복을 없애(setUnion) '들어 있는 리뷰 수'를 센다 → ES와 같은 기준. 불용어도 똑같이 제외. ----
_TOKENS = {'$map': {
    'input': {'$regexFindAll': {'input': {'$ifNull': ['$payload.review', '']}, 'regex': '[가-힣A-Za-z0-9]{2,}'}},
    'as': 'm', 'in': {'$toLower': '$$m.match'}}}

def mongo_keywords(appid: int, k: int):
    pipeline = [
        {'$match': {'payload.steam_appid': appid, 'payload.language': LANG}},
        {'$project': {'_id': 0, 't': {'$setUnion': [_TOKENS, []]}}},
        {'$unwind': '$t'},
        {'$match': {'t': {'$nin': STOPWORDS}}},
        {'$group': {'_id': '$t', 'n': {'$sum': 1}}},
        {'$sort': {'n': -1, '_id': 1}},
        {'$limit': k},
    ]
    return [(d['_id'], d['n']) for d in coll.aggregate(pipeline, allowDiskUse=True, maxTimeMS=MONGO_TIMEOUT_MS)]

# ---- 방법 3(선택): Mongo에서 리뷰를 꺼내 와 Kiwi로 명사 추출 ----
try:
    from kiwipiepy import Kiwi
    kiwi = Kiwi()
    HAS_KIWI = True
except Exception as e:                      # 설치 안 됨, 또는 이 Python 버전용 빌드가 없음
    kiwi, HAS_KIWI = None, False
    print('kiwipiepy 사용 불가 → Mongo+Kiwi 비교는 건너뜁니다:', type(e).__name__)

NOUN_TAGS = {'NNG', 'NNP'}
def kiwi_keywords(appid: int, k: int):
    """가져온 리뷰 수(n_used)도 함께 돌려준다. MAX_FETCH보다 많으면 일부만 쓴 것."""
    docs = coll.find({'payload.steam_appid': appid, 'payload.language': LANG},
                     {'_id': 0, 'payload.review': 1}, max_time_ms=MONGO_TIMEOUT_MS).limit(MAX_FETCH)
    texts = [(d.get('payload') or {}).get('review') or '' for d in docs]
    cnt = Counter()
    for text in texts:
        words = {t.form.lower() for t in kiwi.tokenize(text) if t.tag in NOUN_TAGS and len(t.form) >= 2}
        cnt.update(w for w in words if w not in STOPWORDS)
    return cnt.most_common(k), len(texts)

print('추출 함수 준비 완료. Kiwi:', '사용' if HAS_KIWI else '미사용')

In [ ]:
# ---- 품질 지표: 상위 K개 중 "조사·어미가 붙은 어절로 의심되는" 비율 ----
# 규칙: 키워드에서 끝의 조사·어미를 떼었을 때 남는 어간(2글자 이상)이 같은 게임의 어휘(상위 VOCAB_K)에 따로 있으면 의심.
#       예) '캐릭터가'는 어휘에 '캐릭터'가 있으니 의심, '플레이'는 '플레'가 어휘에 없으니 정상.
#       어휘에는 불용어도 들어간다('게임이'의 어간 '게임'은 불용어라 결과에서는 빠지지만 어간으로는 인정).
#       ES·Mongo 모두 같은 규칙·같은 어휘로 판정한다.
SUFFIXES = ['이다', '입니다', '에서', '으로', '하고', '해서', '네요', '어요', '아요', '지만', '는데',
            '이', '가', '은', '는', '을', '를', '에', '의', '도', '로', '와', '과', '만']

def suspect(token: str, vocab: set) -> bool:
    for s in SUFFIXES:
        if token.endswith(s) and len(token) - len(s) >= 2 and token[:-len(s)] in vocab:
            return True
    return False

def noise_rate(tokens, vocab: set) -> float:
    return float(np.mean([suspect(t, vocab) for t in tokens])) if tokens else np.nan

# Kiwi가 있으면 독립된 두 번째 판정: 키워드를 Kiwi로 분석했을 때 전부 명사 계열이면 '순수'
PURE_TAGS = {'NNG', 'NNP', 'XSN', 'SL'}
def kiwi_impure_rate(tokens) -> float:
    if not HAS_KIWI or not tokens:
        return np.nan
    def impure(t):
        parts = kiwi.tokenize(t)
        return not (parts and all(p.tag in PURE_TAGS for p in parts))
    return float(np.mean([impure(t) for t in tokens]))

# 간단한 동작 확인 (가짜 어휘)
_v = {'캐릭터', '스토리'}
print('의심 판정 예:', {t: suspect(t, _v) for t in ['캐릭터가', '스토리를', '플레이', '캐릭터']})

## 5. 측정 실행

게임마다 세 방법을 **같은 순서로** 실행합니다. 첫 실행은 버리고(워밍업) `REPEATS`번의 **중앙값**을 씁니다.
Kiwi는 느려서 한 번만 잽니다. 시간이 오래 걸리면 중간 진행 상황이 출력됩니다.

In [ ]:
def timed(fn, *args):
    t0 = time.perf_counter()
    out = fn(*args)
    return out, (time.perf_counter() - t0) * 1000

def median_ms(fn, appid, k):
    last, times = None, []
    for i in range(WARMUP + REPEATS):
        try:
            last, ms = timed(fn, appid, k)
        except Exception as e:                       # 시간 초과 등
            print(f'   ! {fn.__name__}({appid}) 실패: {type(e).__name__}')
            return None, np.nan
        if i >= WARMUP:
            times.append(ms)
    return last, float(np.median(times))

rows, tops, vocabs = [], {}, {}
t_start = time.perf_counter()
for i, g in games.iterrows():
    a, n = int(g['steam_appid']), int(g['n_es'])
    es_top, es_ms = median_ms(es_keywords, a, TOP_K_MAX)
    mg_top, mg_ms = median_ms(mongo_keywords, a, TOP_K_MAX)
    # 어휘 (측정 시간에 포함하지 않음)
    # 불용어('게임', '그냥' …)도 어간으로 인정해야 '게임이', '그냥도' 같은 결합형이 잡힌다
    try:
        vocab = {t for t, _ in es_keywords(a, VOCAB_K)} | {t for t, _ in mongo_keywords(a, VOCAB_K)} | set(STOPWORDS)
    except Exception:
        vocab = set(STOPWORDS)
    kw_top, kw_ms, kw_n = None, np.nan, np.nan
    if HAS_KIWI:
        try:
            (kw_top, kw_n), kw_ms = timed(kiwi_keywords, a, TOP_K_MAX)
        except Exception as e:
            print(f'   ! kiwi({a}) 실패: {type(e).__name__}')
    tops[a] = {'es': es_top, 'mongo': mg_top, 'kiwi': kw_top}
    vocabs[a] = vocab
    rows.append({'steam_appid': a, 'n_reviews': n, 'es_ms': es_ms, 'mongo_ms': mg_ms, 'kiwi_ms': kw_ms, 'kiwi_n': kw_n})
    print(f'[{i+1:>2}/{len(games)}] appid {a:<8} 리뷰 {n:>6,}건  ES {es_ms:8.1f}ms  Mongo {mg_ms:9.1f}ms'
          + (f'  Kiwi {kw_ms:9.1f}ms' if HAS_KIWI else '') + f'   (경과 {time.perf_counter()-t_start:.0f}s)')

df = pd.DataFrame(rows).dropna(subset=['es_ms', 'mongo_ms']).reset_index(drop=True)
print(f'\n측정 완료: {len(df)}개 게임')
display(df.head())

In [ ]:
# 지표 계산: 게임마다 상위 K개의 잡음 비율(K=10·20·50), 두 방법의 겹침(Jaccard), Kiwi 판정
def top_tokens(a, who, k):
    t = tops[a][who]
    return [x for x, _ in (t or [])][:k]

recs = []
for _, r in df.iterrows():
    a = int(r['steam_appid']); v = vocabs[a]
    d = {'steam_appid': a}
    for k in (10, 20, 50):
        d[f'noise_es_{k}'] = noise_rate(top_tokens(a, 'es', k), v)
        d[f'noise_mongo_{k}'] = noise_rate(top_tokens(a, 'mongo', k), v)
        if tops[a]['kiwi']:
            d[f'noise_kiwi_{k}'] = noise_rate(top_tokens(a, 'kiwi', k), v)
    es_k, mg_k = set(top_tokens(a, 'es', TOP_K)), set(top_tokens(a, 'mongo', TOP_K))
    d['jaccard_es_mongo'] = len(es_k & mg_k) / max(len(es_k | mg_k), 1)
    if HAS_KIWI:
        d['impure_es'] = kiwi_impure_rate(top_tokens(a, 'es', TOP_K))
        d['impure_mongo'] = kiwi_impure_rate(top_tokens(a, 'mongo', TOP_K))
    recs.append(d)
df = df.merge(pd.DataFrame(recs), on='steam_appid')
df['ratio_mongo_es'] = df['mongo_ms'] / df['es_ms']
display(df[['steam_appid', 'n_reviews', 'es_ms', 'mongo_ms', 'ratio_mongo_es',
            f'noise_es_{TOP_K}', f'noise_mongo_{TOP_K}', 'jaccard_es_mongo']].describe().T)

## 6. 탐색 (EDA): 눈으로 먼저 보기

### 6-1. 같은 게임, 두 방법의 키워드 (가장 리뷰가 많은 게임)

In [ ]:
big = int(df.sort_values('n_reviews').iloc[-1]['steam_appid'])
v = vocabs[big]
N_SHOW = 15
fig, axes = plt.subplots(1, 2, figsize=(11, 5.2), sharex=False)
for ax, who, color, title in [(axes[0], 'es', BLUE, 'ES (nori 명사)'), (axes[1], 'mongo', ORANGE, 'Mongo (집계, 형태소 분석 없음)')]:
    items = (tops[big][who] or [])[:N_SHOW]
    names = [t for t, _ in items]; vals = [c for _, c in items]
    colors = [GRAY if suspect(t, v) else color for t in names]       # 조사 결합 의심은 회색
    ax.barh(range(len(names))[::-1], vals, color=colors)
    ax.set_yticks(range(len(names))[::-1]); ax.set_yticklabels(names)
    ax.set_title(title); ax.set_xlabel('그 단어가 들어 있는 리뷰 수')
fig.suptitle(f'appid {big}: 상위 {N_SHOW}개 키워드 (회색 = 조사·어미 결합 의심)', y=1.02)
plt.tight_layout(); plt.savefig(RESULT_DIR / 'bench_keywords.png', dpi=130, bbox_inches='tight'); plt.show()

side = pd.DataFrame({'ES': [t for t, _ in (tops[big]['es'] or [])[:N_SHOW]],
                     'Mongo(집계)': [t for t, _ in (tops[big]['mongo'] or [])[:N_SHOW]]})
display(side)

#### 차트 해설 / 이 차트로 알 수 있는 것
* 막대 길이는 *그 단어가 들어 있는 리뷰 수*이고, **회색 막대**는 4장의 규칙으로 "조사·어미가 붙은 어절일 가능성"이 있다고 본 단어입니다.
* 아래 표에서 두 목록을 나란히 눈으로 읽어 보세요. **"같은 단어가 조사만 달리해 여러 줄로 갈라져 있는가"**, **"의미 있는 단어(그래픽, 스토리 …)가 상위에 있는가"** 가 핵심입니다. 숫자 통계(7장)는 이 눈대중을 확인하는 용도입니다.
* 한 게임만 본 것이므로 일반화하지 않습니다.

### 6-2. 시간: 리뷰 수에 따라 어떻게 늘어나는가

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
ax = axes[0]
ax.scatter(df['n_reviews'], df['es_ms'], color=BLUE, label='ES')
ax.scatter(df['n_reviews'], df['mongo_ms'], color=ORANGE, label='Mongo(집계)')
if HAS_KIWI and df['kiwi_ms'].notna().any():
    ax.scatter(df['n_reviews'], df['kiwi_ms'], color=GREEN, label='Mongo+Kiwi', marker='^')
ax.set_xscale('log'); ax.set_yscale('log'); plain_log(ax, 'x'); plain_log(ax, 'y')
ax.set_xlabel('게임의 한국어 리뷰 수 (로그)'); ax.set_ylabel('키워드 추출 시간 ms (로그)')
ax.set_title('리뷰 수 vs 추출 시간'); ax.legend()

ax = axes[1]
data = [df['es_ms'], df['mongo_ms']] + ([df['kiwi_ms'].dropna()] if HAS_KIWI and df['kiwi_ms'].notna().any() else [])
labels = ['ES', 'Mongo(집계)'] + (['Mongo+Kiwi'] if len(data) == 3 else [])
bp = ax.boxplot(data, patch_artist=True)
ax.set_xticks(range(1, len(labels) + 1)); ax.set_xticklabels(labels)
for patch, c in zip(bp['boxes'], [BLUE, ORANGE, GREEN]):
    patch.set_facecolor(c); patch.set_alpha(.6)
ax.set_yscale('log'); plain_log(ax, 'y'); ax.set_ylabel('시간 ms (로그)'); ax.set_title('게임별 추출 시간 분포')
plt.tight_layout(); plt.savefig(RESULT_DIR / 'bench_time.png', dpi=130); plt.show()

#### 차트 해설 / 이 차트로 알 수 있는 것
* **왼쪽**: 점이 오른쪽 위로 가파르게 올라가면 "리뷰가 많을수록 시간이 급격히 늘어난다"는 뜻입니다. ES 점이 평평하면 규모와 거의 무관하다는 뜻이고요. 두 축이 모두 로그라서 **직선의 기울기**가 늘어나는 속도입니다. (7장 H3가 이 기울기를 비교합니다.)
* **오른쪽**: 게임들의 시간 분포. 상자가 아래에 있고 폭이 좁을수록 빠르고 안정적입니다.
* 두 방법 모두 **같은 PC에서 잰 왕복 시간**이라 네트워크 지연(수 ms~수십 ms)이 바닥으로 깔려 있습니다. 시간이 그 바닥에 붙어 있다면 "차이가 안 난다"는 뜻일 수 있습니다.

### 6-3. 품질: 가짜 키워드 비율

In [ ]:
k = TOP_K
long = pd.concat([
    pd.DataFrame({'방법': 'ES', '잡음 비율': df[f'noise_es_{k}']}),
    pd.DataFrame({'방법': 'Mongo(집계)', '잡음 비율': df[f'noise_mongo_{k}']}),
] + ([pd.DataFrame({'방법': 'Mongo+Kiwi', '잡음 비율': df[f'noise_kiwi_{k}']})] if f'noise_kiwi_{k}' in df else []))
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))
sns.boxplot(data=long, x='방법', y='잡음 비율', palette=[BLUE, ORANGE, GREEN][:long['방법'].nunique()], ax=axes[0])
sns.stripplot(data=long, x='방법', y='잡음 비율', color='k', size=3, alpha=.5, ax=axes[0])
axes[0].set_title(f'상위 {k}개 중 조사·어미 결합 의심 비율'); axes[0].set_ylim(-0.02, 1)
d = df[f'noise_mongo_{k}'] - df[f'noise_es_{k}']
axes[1].hist(d, bins=12, color=ORANGE, alpha=.8)
axes[1].axvline(0, color='k', lw=1); axes[1].axvline(PRACTICAL_NOISE, color=GRAY, ls='--', label=f'실무 기준 {PRACTICAL_NOISE:.0%}p')
axes[1].set_title('게임별 잡음 비율 차이 (Mongo - ES)'); axes[1].set_xlabel('차이 (+이면 ES가 깨끗)'); axes[1].legend()
plt.tight_layout(); plt.savefig(RESULT_DIR / 'bench_quality.png', dpi=130); plt.show()

#### 차트 해설 / 이 차트로 알 수 있는 것
* **왼쪽**: 방법별로 "상위 키워드 중 가짜 비율"을 게임마다 점으로 찍은 것입니다. 상자가 낮을수록 깨끗합니다.
* **오른쪽**: 같은 게임에서 *Mongo − ES* 차이입니다. **0 오른쪽**이면 ES가 더 깨끗하고, 점선(20%p)보다 오른쪽이면 "실무적으로 의미 있는" 차이입니다. 막대가 0 근처에 몰려 있으면 두 방법이 비슷하다는 뜻입니다.
* 이 비율은 4장의 **규칙 기반 추정치**입니다(정확한 정답표가 아님). 그래서 Kiwi가 있으면 7장에서 독립된 판정으로 한 번 더 확인합니다.

## 7. 통계 검정

공통으로 쓰는 도구를 먼저 정의합니다. 게임이 독립 단위이므로 일반 **부트스트랩**(게임을 다시 뽑기)으로 신뢰구간을 구합니다.

In [ ]:
def boot_ci(x, func=np.median, n=N_BOOT, alpha=0.05):
    x = np.asarray(x, dtype=float); x = x[~np.isnan(x)]
    if len(x) < 3:
        return (np.nan, np.nan)
    stat = [func(rng.choice(x, size=len(x), replace=True)) for _ in range(n)]
    return tuple(np.quantile(stat, [alpha / 2, 1 - alpha / 2]))

def rank_biserial(d):
    """짝 비교 효과크기(−1~+1). +1에 가까우면 거의 모든 게임에서 한쪽이 큼."""
    d = np.asarray(d, dtype=float); d = d[(d != 0) & ~np.isnan(d)]
    if len(d) == 0:
        return 0.0
    ranks = stats.rankdata(np.abs(d))
    return float((ranks[d > 0].sum() - ranks[d < 0].sum()) / ranks.sum())

def paired_test(d, alternative='greater'):
    """d = 짝 차이. 정규성(Shapiro) → Wilcoxon 부호순위(단측)와 양측 p."""
    d = np.asarray(d, dtype=float); d = d[~np.isnan(d)]
    out = {'n': int(len(d)), 'median': float(np.median(d)) if len(d) else np.nan,
           'share_pos': float(np.mean(d > 0)) if len(d) else np.nan}
    out['shapiro_p'] = float(stats.shapiro(d).pvalue) if len(d) >= 3 and np.ptp(d) > 0 else np.nan
    if len(d) >= 6 and np.ptp(d) > 0:
        out['p_one'] = float(stats.wilcoxon(d, alternative=alternative).pvalue)
        out['p_two'] = float(stats.wilcoxon(d, alternative='two-sided').pvalue)
    else:
        out['p_one'] = out['p_two'] = np.nan
    out['ci'] = boot_ci(d)
    out['r'] = rank_biserial(d)
    return out

def verdict(res, practical, higher_is_hypothesis=True):
    """가설 방향(+)·유의·실무 기준을 모두 보고 한 줄 판정을 만든다."""
    if np.isnan(res['p_adj']):
        return '표본 부족'
    if res['median'] < 0 and res['p_two'] < ALPHA:
        return '불일치 (반대 방향으로 유의)'
    if res['p_adj'] < ALPHA and res['median'] >= practical:
        return '일치 · 의미 있음'
    if res['p_adj'] < ALPHA:
        return '일치(방향) · 효과가 작아 의미는 약함'
    return '불일치 (차이가 확인되지 않음)'

R = {'n_games': int(len(df)), 'mongo_has_index': bool(mongo_has_index), 'has_kiwi': bool(HAS_KIWI)}

### H1. 품질: ES가 가짜 키워드가 더 적은가
* 차이 = (Mongo 잡음 비율) − (ES 잡음 비율). **+** 이면 ES가 깨끗합니다.

In [ ]:
d1 = (df[f'noise_mongo_{TOP_K}'] - df[f'noise_es_{TOP_K}'])
h1 = paired_test(d1, 'greater')
print('[H1] 상위', TOP_K, '개 기준, 게임', h1['n'], '개')
print(f"  정규성 Shapiro p = {h1['shapiro_p']:.3f}  → {'정규성 의심' if h1['shapiro_p'] < ALPHA else '정규성 기각 못함'} (그래도 순위 기반 Wilcoxon을 씁니다: 표본이 작고 0~1 비율이라)")
print(f"  차이 중앙값 = {h1['median']:+.1%}p   95% 신뢰구간 [{h1['ci'][0]:+.1%}, {h1['ci'][1]:+.1%}]")
print(f"  ES가 더 깨끗한 게임 비율 = {h1['share_pos']:.0%}   효과크기 r(순위쌍상관) = {h1['r']:+.2f}")
print(f"  Wilcoxon 단측 p = {h1['p_one']:.4g} (양측 {h1['p_two']:.4g})")

# Kiwi 독립 판정: 명사로 분석되지 않는 키워드의 비율
h1k = None
if HAS_KIWI and 'impure_es' in df and df['impure_es'].notna().any():
    dk = df['impure_mongo'] - df['impure_es']
    h1k = paired_test(dk, 'greater')
    print(f"\n[H1 확인: Kiwi 독립 판정] 차이 중앙값 {h1k['median']:+.1%}p, 신뢰구간 [{h1k['ci'][0]:+.1%}, {h1k['ci'][1]:+.1%}], p={h1k['p_one']:.4g}")
else:
    print('\n(Kiwi가 없어 독립 판정은 하지 않았습니다. 4장 규칙 기반 지표만으로 판단합니다.)')

### H2. 속도: ES가 더 빠른가
* 시간 비율은 곱셈 관계라서 **로그 비율**(log₂(Mongo ÷ ES))로 검정합니다. +이면 Mongo가 느립니다.

In [ ]:
lr = np.log2(df['mongo_ms'] / df['es_ms'])
h2 = paired_test(lr, 'greater')
ratio_med = 2 ** h2['median']; ratio_ci = (2 ** h2['ci'][0], 2 ** h2['ci'][1])
print('[H2] 게임', h2['n'], '개')
print(f"  Shapiro p = {h2['shapiro_p']:.3f}")
print(f"  Mongo가 ES보다 느린 정도(중앙값) = {ratio_med:.2f}배   95% 신뢰구간 [{ratio_ci[0]:.2f}, {ratio_ci[1]:.2f}]배")
print(f"  Mongo가 더 느린 게임 비율 = {h2['share_pos']:.0%}   효과크기 r = {h2['r']:+.2f}")
print(f"  Wilcoxon 단측 p = {h2['p_one']:.4g} (양측 {h2['p_two']:.4g})")
print(f"  중앙값: ES {df['es_ms'].median():.0f}ms, Mongo {df['mongo_ms'].median():.0f}ms" + (f", Kiwi {df['kiwi_ms'].median():.0f}ms" if HAS_KIWI and df['kiwi_ms'].notna().any() else ''))

### H3. 규모: 리뷰가 많을수록 Mongo가 더 가파르게 느려지는가
* `log10(시간) = 기울기 × log10(리뷰 수) + 절편`. 기울기 1이면 "리뷰가 2배 되면 시간도 2배", 0이면 "규모와 무관".
* 기울기 차이(Mongo − ES)의 신뢰구간은 **게임을 다시 뽑는 부트스트랩**으로 구합니다.

In [ ]:
x = np.log10(df['n_reviews'].to_numpy(dtype=float))
def fit(xv, yv):
    res = stats.linregress(xv, np.log10(yv))
    return res.slope, res.intercept, res.rvalue ** 2

if len(df) >= 8 and np.ptp(x) > 0.3:
    s_es, b_es, r2_es = fit(x, df['es_ms'].to_numpy())
    s_mg, b_mg, r2_mg = fit(x, df['mongo_ms'].to_numpy())
    diffs = []
    for _ in range(N_BOOT):
        idx = rng.integers(0, len(df), len(df))
        if np.ptp(x[idx]) < 1e-9:
            continue
        diffs.append(fit(x[idx], df['mongo_ms'].to_numpy()[idx])[0] - fit(x[idx], df['es_ms'].to_numpy()[idx])[0])
    ci = tuple(np.quantile(diffs, [.025, .975]))
    rho_es = stats.spearmanr(df['n_reviews'], df['es_ms'])
    rho_mg = stats.spearmanr(df['n_reviews'], df['mongo_ms'])
    h3 = {'slope_es': s_es, 'slope_mongo': s_mg, 'r2_es': r2_es, 'r2_mongo': r2_mg, 'diff': s_mg - s_es, 'ci': ci,
          'rho_es': float(rho_es.statistic), 'rho_mongo': float(rho_mg.statistic),
          'p_es': float(rho_es.pvalue), 'p_mongo': float(rho_mg.pvalue)}
    print('[H3] 로그-로그 기울기 (시간 ~ 리뷰 수)')
    print(f"  ES    : 기울기 {s_es:+.2f}  (R²={r2_es:.2f})   Spearman ρ={h3['rho_es']:+.2f} (p={h3['p_es']:.3g})")
    print(f"  Mongo : 기울기 {s_mg:+.2f}  (R²={r2_mg:.2f})   Spearman ρ={h3['rho_mongo']:+.2f} (p={h3['p_mongo']:.3g})")
    print(f"  기울기 차이(Mongo−ES) = {h3['diff']:+.2f}   95% 신뢰구간 [{ci[0]:+.2f}, {ci[1]:+.2f}]")
else:
    h3 = None
    print('[H3] 표본 수 또는 리뷰 수 범위가 부족해 검정하지 않습니다 (게임 8개 이상, 리뷰 수 범위 2배 이상 필요).')

In [ ]:
# H3 시각화: 로그-로그 회귀선
if h3:
    fig, ax = plt.subplots(figsize=(7.5, 4.6))
    xs = np.linspace(x.min(), x.max(), 50)
    ax.scatter(10 ** x, df['es_ms'], color=BLUE, label=f"ES (기울기 {h3['slope_es']:+.2f})")
    ax.scatter(10 ** x, df['mongo_ms'], color=ORANGE, label=f"Mongo (기울기 {h3['slope_mongo']:+.2f})")
    ax.plot(10 ** xs, 10 ** (b_es + h3['slope_es'] * xs), color=BLUE)
    ax.plot(10 ** xs, 10 ** (b_mg + h3['slope_mongo'] * xs), color=ORANGE)
    ax.set_xscale('log'); ax.set_yscale('log'); plain_log(ax, 'x'); plain_log(ax, 'y')
    ax.set_xlabel('게임의 한국어 리뷰 수 (로그)'); ax.set_ylabel('추출 시간 ms (로그)')
    ax.set_title('규모 증가에 따른 시간 (회귀선)'); ax.legend()
    plt.tight_layout(); plt.savefig(RESULT_DIR / 'bench_scaling.png', dpi=130); plt.show()

#### 차트 해설 / 이 차트로 알 수 있는 것
* 선이 가파를수록 *규모가 커질 때 시간이 빨리 늘어납니다.* 두 선이 **벌어지면** 큰 게임에서 격차가 더 커진다는 뜻이고, **평행하면** 규모와 상관없이 비슷한 배수 차이입니다.
* 범례의 기울기가 핵심입니다. 두 기울기가 모두 0 근처라면 시간이 네트워크 지연에 묻혀 있다는 뜻이라, 규모 효과를 이 방법으로는 구분하기 어렵습니다.
* 게임 수가 적으면 기울기가 크게 흔들립니다. 신뢰구간이 0을 포함하면 "차이가 있다"고 말할 수 없습니다.

### 다중 비교 보정과 판정 (H1~H3)
세 가설을 동시에 검정했으므로 **Holm 보정**한 p로 판정합니다.

In [ ]:
ps = [h1['p_one'], h2['p_one'], np.nan]
if h3:
    # H3의 p: 부트스트랩 기울기 차이가 0 이하일 비율(단측)
    ps[2] = float((np.array(diffs) <= 0).mean())
valid = [i for i, p in enumerate(ps) if not np.isnan(p)]
adj = [np.nan] * 3
if valid:
    adjusted = multipletests([ps[i] for i in valid], method='holm')[1]
    for i, a in zip(valid, adjusted):
        adj[i] = float(a)
h1['p_adj'], h2['p_adj'] = adj[0], adj[1]
h1['verdict'] = verdict(h1, PRACTICAL_NOISE)
h2_eff = dict(h2); h2_eff['median'] = ratio_med        # 실무 기준은 '배수'로 비교
h2['verdict'] = ('불일치 (반대 방향으로 유의)' if (h2['median'] < 0 and h2['p_two'] < ALPHA) else
                 '표본 부족' if np.isnan(h2['p_adj']) else
                 '일치 · 의미 있음' if (h2['p_adj'] < ALPHA and ratio_med >= PRACTICAL_SPEED) else
                 '일치(방향) · 효과가 작아 의미는 약함' if h2['p_adj'] < ALPHA else
                 '불일치 (차이가 확인되지 않음)')
if h3:
    h3['p_adj'] = adj[2]
    h3['verdict'] = ('표본 부족' if np.isnan(adj[2]) else
                     '일치 · 의미 있음' if (adj[2] < ALPHA and h3['diff'] >= PRACTICAL_SLOPE and h3['ci'][0] > 0) else
                     '일치(방향) · 효과가 작아 의미는 약함' if adj[2] < ALPHA else
                     '불일치 (차이가 확인되지 않음)')
R.update({'h1': h1, 'h2': {**h2, 'ratio_median': ratio_med, 'ratio_ci': ratio_ci}, 'h3': h3})

summary = pd.DataFrame([
    ['H1 품질 (ES가 더 깨끗)', f"{h1['median']:+.1%}p", f"[{h1['ci'][0]:+.1%}, {h1['ci'][1]:+.1%}]", h1['p_adj'], f"≥ {PRACTICAL_NOISE:.0%}p", h1['verdict']],
    ['H2 속도 (ES가 더 빠름)', f"{ratio_med:.2f}배", f"[{ratio_ci[0]:.2f}, {ratio_ci[1]:.2f}]", h2['p_adj'], f"≥ {PRACTICAL_SPEED}배", h2['verdict']],
    ['H3 규모 (Mongo가 더 가파름)', (f"{h3['diff']:+.2f}" if h3 else '–'), (f"[{h3['ci'][0]:+.2f}, {h3['ci'][1]:+.2f}]" if h3 else '–'),
     (h3['p_adj'] if h3 else np.nan), f"≥ {PRACTICAL_SLOPE}", (h3['verdict'] if h3 else '표본 부족')],
], columns=['가설', '효과(중앙값/차이)', '95% 신뢰구간', 'Holm p', '실무 기준', '판정'])
display(summary)

### H4 (참고·탐색): Mongo+Kiwi는 ES와 품질이 비슷한가
*Kiwi가 있을 때만 실행합니다.* 이 비교는 사전에 세운 가설이 아니라 **"Mongo만으로 안 되는 게 아니라, 형태소 분석기만 있으면 된다"는 반론에 답하기 위한 탐색**입니다. 차이의 95% 신뢰구간이 ±10%p 안이면 "비슷하다"고 봅니다.

In [ ]:
h4 = None
if HAS_KIWI and f'noise_kiwi_{TOP_K}' in df and df[f'noise_kiwi_{TOP_K}'].notna().sum() >= 6:
    d4 = df[f'noise_kiwi_{TOP_K}'] - df[f'noise_es_{TOP_K}']
    h4 = paired_test(d4, 'two-sided')
    h4['similar'] = bool(h4['ci'][0] >= -EQUIV and h4['ci'][1] <= EQUIV)
    lr4 = np.log2(df['kiwi_ms'].dropna() / df.loc[df['kiwi_ms'].notna(), 'es_ms'])
    h4['time_ratio_kiwi_es'] = float(2 ** np.median(lr4))
    print(f"[H4] 잡음 비율 차이(Kiwi − ES) 중앙값 {h4['median']:+.1%}p, 95% 신뢰구간 [{h4['ci'][0]:+.1%}, {h4['ci'][1]:+.1%}], 양측 p={h4['p_two']:.3g}")
    print(f"     → {'품질은 비슷함 (±10%p 안)' if h4['similar'] else '품질 차이가 있음 (또는 불확실)'}")
    print(f"     시간: Kiwi 방식은 ES의 약 {h4['time_ratio_kiwi_es']:.1f}배 (데이터를 꺼내 와 분석하는 시간 포함)")
else:
    print('(Kiwi 비교 없음)')
R['h4'] = h4

## 8. 결론이 조건에 따라 흔들리는가 (민감도)

한 조건에서만 나온 결과는 믿기 어렵습니다. **키워드 개수(K)** 와 **가장 큰 게임을 뺐을 때**로 다시 확인합니다.

In [ ]:
rows = []
for k in (10, 20, 50):
    d = (df[f'noise_mongo_{k}'] - df[f'noise_es_{k}']).dropna()
    t = paired_test(d, 'greater')
    rows.append({'조건': f'상위 {k}개', '게임 수': t['n'], '잡음 차이 중앙값': t['median'], '신뢰구간 하한': t['ci'][0], '신뢰구간 상한': t['ci'][1], 'p(단측)': t['p_one']})
no_big = df[df['n_reviews'] < df['n_reviews'].max()]
d = (no_big[f'noise_mongo_{TOP_K}'] - no_big[f'noise_es_{TOP_K}']).dropna()
t = paired_test(d, 'greater')
rows.append({'조건': f'가장 큰 게임 제외 (K={TOP_K})', '게임 수': t['n'], '잡음 차이 중앙값': t['median'], '신뢰구간 하한': t['ci'][0], '신뢰구간 상한': t['ci'][1], 'p(단측)': t['p_one']})
sens = pd.DataFrame(rows)
display(sens)
R['sensitivity'] = sens.to_dict('records')

fig, ax = plt.subplots(figsize=(8, 3.6))
y = np.arange(len(sens))[::-1]
ax.errorbar(sens['잡음 차이 중앙값'], y, xerr=[sens['잡음 차이 중앙값'] - sens['신뢰구간 하한'], sens['신뢰구간 상한'] - sens['잡음 차이 중앙값']],
            fmt='o', color=BLUE, capsize=4)
ax.axvline(0, color='k', lw=1); ax.axvline(PRACTICAL_NOISE, color=GRAY, ls='--', label=f'실무 기준 {PRACTICAL_NOISE:.0%}p')
ax.set_yticks(y); ax.set_yticklabels(sens['조건']); ax.set_xlabel('잡음 비율 차이 (Mongo - ES), 95% 신뢰구간'); ax.legend()
ax.set_title('조건을 바꿔도 같은 결론인가')
plt.tight_layout(); plt.savefig(RESULT_DIR / 'bench_sensitivity.png', dpi=130); plt.show()

#### 차트 해설 / 이 차트로 알 수 있는 것
* 점은 효과 크기, 가로선은 95% 신뢰구간입니다. **모든 선이 0의 오른쪽**에 있으면 조건이 달라도 "ES가 더 깨끗하다"는 결론이 유지됩니다.
* 점선(20%p) 오른쪽까지 가야 실무적으로 의미 있는 크기입니다. 조건에 따라 0을 걸치면 결론이 불안정하다는 뜻이라 발표에서 단정하지 않아야 합니다.

## 9. 결론과 제안 — "왜 Elasticsearch인가"

아래 문장은 **7~8장의 실제 계산값**으로 만들어집니다. 결과가 가설과 반대면 문장도 그대로 반대로 나옵니다.

In [ ]:
L = []
f = lambda h: h['verdict']
L.append('■ 가설 판정')
L.append(f"  H1 품질  : {f(h1)}  (잡음 비율 차이 중앙값 {h1['median']:+.1%}p, 신뢰구간 [{h1['ci'][0]:+.1%}, {h1['ci'][1]:+.1%}])")
L.append(f"  H2 속도  : {f(h2)}  (Mongo가 ES의 {ratio_med:.2f}배, 신뢰구간 [{ratio_ci[0]:.2f}, {ratio_ci[1]:.2f}]배)")
L.append(f"  H3 규모  : {h3['verdict'] if h3 else '표본 부족'}" + (f"  (기울기 ES {h3['slope_es']:+.2f}, Mongo {h3['slope_mongo']:+.2f})" if h3 else ''))
if h4:
    L.append(f"  (탐색) Mongo+Kiwi 품질 : {'ES와 비슷함' if h4['similar'] else 'ES와 차이 있음 또는 불확실'} (차이 {h4['median']:+.1%}p, 시간은 ES의 약 {h4['time_ratio_kiwi_es']:.1f}배)")

q_ok = h1['verdict'].startswith('일치 ·')
s_ok = h2['verdict'].startswith('일치 ·')

L.append('\n■ "왜 ES인가" — 발표에서 말할 수 있는 것')
if q_ok:
    L.append(f"  - ES는 한국어 형태소 분석(nori)이 인덱스 안에서 되어, Mongo 집계보다 조사·어미가 붙은 가짜 키워드가 {h1['median']:.0%}p 적게 나왔다. (게임 {h1['n']}개, 순위 기반 검정, Holm 보정 p={h1['p_adj']:.3g})")
else:
    L.append(f"  - 이 데이터에서는 품질 우위를 입증하지 못했다({h1['verdict']}). '품질이 좋아서 ES'라는 문장은 쓰지 않는다. 불용어·품사 설정을 먼저 손보고 다시 측정한다.")
if s_ok:
    L.append(f"  - 같은 키워드 추출이 ES에서 {ratio_med:.1f}배 빨랐다." + ('  (단, Mongo에 appid 인덱스가 없는 상태 기준이라 인덱스를 만들면 격차가 줄 수 있음)' if not mongo_has_index else ''))
else:
    L.append(f"  - 속도로는 ES를 내세우지 않는다({h2['verdict']}). 키워드 추출은 하루 1회 배치라 속도가 핵심 이유도 아니다.")
if h4 and h4['similar']:
    L.append("  - 정직하게: Mongo 데이터를 꺼내 Kiwi로 분석해도 품질은 ES와 비슷했다. ES의 이점은 '품질'이 아니라 **형태소 분석과 집계가 저장소 안에서 한 번에 끝나 데이터를 밖으로 꺼내 가공할 필요가 없고, 시간도 "
             f"{h4['time_ratio_kiwi_es']:.1f}배 덜 걸린다**는 점이다.")
elif h4:
    L.append("  - Mongo+Kiwi와 비교해도 ES의 품질 차이가 남았다 → 형태소 분석기를 따로 붙이는 방식이 같은 수준을 보장하지는 않았다.")
else:
    L.append("  - 예상되는 반론: 'Mongo에 형태소 분석기를 따로 붙이면 되지 않나?' → Kiwi가 설치되지 않아 이번에는 측정하지 못했다. 설치해 다시 돌리면 4번째 비교(H4)가 채워진다.")
L.append("  - ES를 쓰는 다른 이유(이 노트북 범위 밖): 클릭스트림(검색 로그)의 시간 창 집계와 Kibana 시각화.")

L.append('\n■ 운영 제안')
if not mongo_has_index:
    L.append("  - Mongo `reviews`에 `payload.steam_appid` 인덱스가 없다. 비교를 공정하게 하려면 인덱스를 만든 뒤 다시 측정해 두 결과를 함께 보인다(인덱스 생성은 팀 결정 사항).")
L.append("  - 잡음 의심 키워드(회색)가 ES 결과에도 남아 있다면 `review_stop`(불용어)·`stoptags`(품사)에 추가해 인덱스를 다시 만든다 (infra/monstache/README.md 절차).")
L.append("  - AI 요약에 보내는 키워드는 지금 TOP 10개다. 잡음이 줄수록 같은 토큰 수로 더 많은 의미를 보낸다(AI 비용·품질 모두 이득).")
print('\n'.join(L))

(RESULT_DIR / 'summary.json').write_text(json.dumps(R, ensure_ascii=False, indent=2, default=float), encoding='utf-8')
df.to_csv(RESULT_DIR / 'bench_per_game.csv', index=False, encoding='utf-8-sig')
table = ['| ' + ' | '.join(summary.columns) + ' |', '|' + '---|' * len(summary.columns)]
table += ['| ' + ' | '.join('' if (isinstance(v, float) and np.isnan(v)) else (f'{v:.3g}' if isinstance(v, float) else str(v)) for v in row) + ' |' for row in summary.itertuples(index=False)]
md_lines = ['# ES vs MongoDB 리뷰 키워드 추출 벤치마크 결과', '', f'* 게임 {len(df)}개, 상위 {TOP_K}개 키워드, 반복 {REPEATS}회 중앙값', ''] + table
(RESULT_DIR / 'results.md').write_text('\n'.join(md_lines), encoding='utf-8')
print('\n저장:', RESULT_DIR / 'summary.json', '·', RESULT_DIR / 'results.md', '·', RESULT_DIR / 'bench_per_game.csv')

### 한계 (발표에서 먼저 밝힐 것)
| 한계 | 영향 | 대응 |
| --- | --- | --- |
| 품질 지표가 **규칙 기반 추정**(정답표 없음) | 잡음 비율의 절대값은 부정확, 방법 간 *상대 비교*만 의미 | Kiwi 독립 판정을 함께 보고, 6-1의 나란히 보기로 눈 확인 |
| **왕복 시간**(네트워크 포함) 측정 | 서버 내부 성능 차이가 작으면 묻힘 | 두 방법에 같은 조건. 서버 내부 시간이 필요하면 ES `took`, Mongo `explain`을 별도 확인 |
| Mongo **인덱스 유무**에 따라 속도가 크게 달라짐 | 인덱스 없으면 Mongo 불리 | 3장에서 확인, 결론에 조건 명시, 필요 시 인덱스 생성 후 재측정 |
| ES **캐시**(파일시스템 캐시 등)를 완전히 끌 수는 없음 | ES가 약간 유리할 수 있음 | 요청 캐시는 껐고, 워밍업 후 중앙값 사용 |
| 게임 **N개 표본**, 리뷰 수 범위가 한정 | 기울기(H3) 불확실 | 신뢰구간과 함께 보고, 범위 밖 일반화 금지 |
| 한국어 리뷰만 비교 | 다른 언어는 별도 분석기가 필요 | 범위를 명시 |
| 운영 DB에 읽기 부하 | 사용자 화면이 느려질 수 있음 | `N_GAMES`·`REPEATS`로 조절, 한가한 시간에 실행, 질의 시간 상한(`MONGO_TIMEOUT_MS`) |